# 07 - Results Interpretation

This notebook consolidates the social network and forecasting results.

Research question:

> Can social interaction patterns between Yelp users help predict future business review activity beyond historical review trends alone?

In [1]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "new_orleans"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

METRICS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_metrics.csv"
PREDICTIONS_OUTPUT_PATH = OUTPUTS_DIR / "forecasting_predictions.csv"
GRAPH_SUMMARY_PATH = PROCESSED_DIR / "social_graph_summary.json"
FEATURE_SUMMARY_PATH = PROCESSED_DIR / "forecasting_feature_summary.json"

metrics = pd.read_csv(METRICS_OUTPUT_PATH)
predictions = pd.read_csv(PREDICTIONS_OUTPUT_PATH)
with GRAPH_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    graph_summary = json.load(file)
with FEATURE_SUMMARY_PATH.open("r", encoding="utf-8") as file:
    feature_summary = json.load(file)

metrics.sort_values(["split", "WAPE", "MAE"])

,split,model,rows,MAE,RMSE,WAPE
0,primary_covid_test,Baseline: last month,27624,1.311323,2.653812,0.688473
1,primary_covid_test,Baseline: rolling 3-month avg,27624,1.342685,2.954533,0.704938
2,primary_covid_test,ML: historical,27624,1.445685,3.004346,0.759016
3,primary_covid_test,ML: historical + business + SNA,27624,1.473653,3.049569,0.773699
4,primary_covid_test,ML: historical + business,27624,1.479746,3.077057,0.776898
5,primary_covid_test,Baseline: seasonal naive,27624,2.877643,6.556732,1.510824
6,secondary_pre_covid_test,ML: historical + business,13812,1.833048,3.118196,0.383979
7,secondary_pre_covid_test,ML: historical + business + SNA,13812,1.836260,3.114596,0.384652
8,secondary_pre_covid_test,ML: historical,13812,1.889936,3.156896,0.395896
9,secondary_pre_covid_test,Baseline: rolling 3-month avg,13812,1.954798,3.306649,0.409483


In [2]:
summary_rows = []
for split_name, split_metrics in metrics.groupby("split"):
    ranked = split_metrics.sort_values("WAPE").reset_index(drop=True)
    best = ranked.iloc[0]
    hist = split_metrics[split_metrics["model"] == "ML: historical"].iloc[0]
    hist_business = split_metrics[split_metrics["model"] == "ML: historical + business"].iloc[0]
    sna = split_metrics[split_metrics["model"] == "ML: historical + business + SNA"].iloc[0]
    summary_rows.append({
        "split": split_name,
        "best_model": best["model"],
        "best_WAPE": best["WAPE"],
        "historical_WAPE": hist["WAPE"],
        "historical_business_WAPE": hist_business["WAPE"],
        "sna_WAPE": sna["WAPE"],
        "sna_vs_historical_business_relative_change": (sna["WAPE"] - hist_business["WAPE"]) / hist_business["WAPE"],
        "sna_vs_historical_relative_change": (sna["WAPE"] - hist["WAPE"]) / hist["WAPE"],
    })
interpretation_summary = pd.DataFrame(summary_rows)
interpretation_summary

,split,best_model,best_WAPE,historical_WAPE,historical_business_WAPE,sna_WAPE,sna_vs_historical_business_relative_change,sna_vs_historical_relative_change
0,primary_covid_test,Baseline: last month,0.688473,0.759016,0.776898,0.773699,-0.004117,0.019346
1,secondary_pre_covid_test,ML: historical + business,0.383979,0.395896,0.383979,0.384652,0.001753,-0.028401


In [3]:
print("Social graph summary")
for key, value in graph_summary.items():
    print(f"{key}: {value}")

print("\nForecasting dataset summary")
for key, value in feature_summary.items():
    print(f"{key}: {value}")

Social graph summary
active_review_threshold: 5
reviewing_users: 245421
matched_user_profiles: 245419
active_users: 26598
graph_nodes: 26598
graph_edges: 116558
connected_components: 11508
largest_component_size: 14965
isolated_active_users: 11387
community_method: louvain_largest_component
communities_assigned: 24
output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\user_network_features.csv

Forecasting dataset summary
min_total_reviews: 100
min_active_months: 36
business_count: 1151
row_count: 95533
feature_month_min: 2015-01
feature_month_max: 2021-11
target_month_min: 2015-02
target_month_max: 2021-12
excluded_partial_month: 2022-01
output: C:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\data\processed\new_orleans\forecasting_dataset.csv


In [4]:
for _, row in interpretation_summary.iterrows():
    split = row["split"]
    change_vs_business = row["sna_vs_historical_business_relative_change"] * 100
    change_vs_hist = row["sna_vs_historical_relative_change"] * 100
    direction_business = "improved" if change_vs_business < 0 else "worsened"
    direction_hist = "improved" if change_vs_hist < 0 else "worsened"
    print(f"{split}:")
    print(f"  Best model: {row['best_model']} with WAPE={row['best_WAPE']:.4f}")
    print(f"  SNA model {direction_business} WAPE vs historical+business by {abs(change_vs_business):.2f}%")
    print(f"  SNA model {direction_hist} WAPE vs historical-only by {abs(change_vs_hist):.2f}%")

primary_covid_test:
  Best model: Baseline: last month with WAPE=0.6885
  SNA model improved WAPE vs historical+business by 0.41%
  SNA model worsened WAPE vs historical-only by 1.93%
secondary_pre_covid_test:
  Best model: ML: historical + business with WAPE=0.3840
  SNA model worsened WAPE vs historical+business by 0.18%
  SNA model improved WAPE vs historical-only by 2.84%


## Interpretation Framework

The result should be interpreted in three layers:

1. **Forecastability:** how far simple historical baselines go.
2. **Business metadata value:** whether business category and static characteristics add signal.
3. **SNA value:** whether recent reviewer network structure improves over historical and business-only features.

The SNA model does not need to win for the project to be successful. A non-improvement is still meaningful if the experiment is fair: it may show that short-term review activity is dominated by recent temporal dynamics, business popularity, and external shocks.

## Limitations

- Yelp friendship links are static; friendship formation dates are unavailable.
- Network features are social context, not causal proof of influence.
- The COVID-era test period contains a major external shock.
- The cohort threshold improves reliability but focuses the task on active businesses.
- Review count measures Yelp engagement, not revenue or true customer volume.

## Final Academic Position

This project should be presented as an interpretable multimodal forecasting experiment. It combines temporal behavior, business metadata, and social network structure to evaluate whether community signals improve prediction of future review activity.